# Matched Product Captioning 
Since image quality can matter a lot in accurate identification of products, we wanted to explore captions quality on product marketing images. These images are likely to have no image quality distortions and fall in the training sets of VLMs. 

In [ ]:
import pandas as pd
import os
import sys
import json
import gc
from datetime import datetime

import requests
from PIL import Image
from io import BytesIO
import base64
import io

from tqdm.notebook import tqdm

from dotenv import load_dotenv

load_dotenv()

sys.path.append("../../")

In [ ]:
# load data
# check if we have any labeled data first
captioned_file = "./intermediate/matched-product-captions-llama.json"
matched_product_df_dict = None
if os.path.isfile(captioned_file):
    with open(captioned_file, "r") as f:
        matched_product_df_dict = json.load(f)
else:
    matched_product_df = pd.read_csv("matched-product-input.csv")
    matched_product_df_dict = matched_product_df.to_dict(orient="records")
matched_product_df_dict[0]

## Prompt config
This notebook tests two prompts, currently:
1. The original VLM prompt we used for the ASSETS study
2. A revised prompt that encourages the model to detail product, brand, and details identified.

In [ ]:
from scripts.constants import get_prompt

VLM_ORIG_PROMPT = get_prompt()
VLM_REV_PROMPT = (
    "You are a helpful assistant who identififes products in images for blind and low-vision individuals. Follow these guidelines and only output the final description:\n"
    "Step 1: Identify the product in the image.\n"
    "Step 2: Identify crucial features about the product from Step 1, including:\n"
    "-- Product, such as seasoning mix, soda, coffee, etc.\n"
    "-- Brand, such as Heinz, Coca-Cola, Starbucks, etc.\n"
    "-- Variety, such as specific flavors, sizes, count of items, etc.\n"
    "-- Visual features, such as color, shape, size, etc.\n"
    "Step 3: Use clear, direct, and objective language. Do not use vague adjectives like 'large' or 'small', and vague adverbs like 'prominently' or 'clearly'.\n"
    "Step 4: DO NOT mention camera artifacts (e.g., blur) or if an object is partially visible.\n"
    "Step 5: DO NOT use introductory phrases (e.g., 'The image shows', 'The object is', 'The primary object is').\n\n"
    "Output only the final description."
)
VLM_REV_PROMPT_2 = (
    "You are an assistant who identifies products in images for blind and low-vision users. Follow these guidelines and then output only the final description:\n"
    "1. Identify the product (e.g., seasoning mix, soda, coffee).\n"
    "2. Extract and label key attributes:\n"
    "-- Product type (what it is).\n"
    "-- Brand (e.g., Heinz, Coca-Cola, Starbucks).\n"
    "-- Variety or variant (flavor, size, count, etc.).\n"
    "-- Visual features (color, shape, distinctive markings).\n"
    "3. Use precise, objective language. Avoid vague descriptors (“large,” “small,” “prominently,” “clearly”).\n"
    "4. DO NOT mention camera artifacts (blur, glare) or cropping/partial visibility.\n"
    "5. DO NOT prepend with “The image shows,” “The object is,” or similar phrases.\n\n"
    "Output only the final description."
)

print("Original Prompt")
print(VLM_ORIG_PROMPT)
print("-" * 100)
print("Revised Prompt 1")
print(VLM_REV_PROMPT)
print("-" * 100)
print("Revised Prompt 2")
print(VLM_REV_PROMPT_2)

## Add captions for all prompts

In [ ]:
def remove_transparency(im, bg_colour=(255, 255, 255)):
    """
    Remove transparency from an image.

    Args:
        im (PIL.Image.Image): Image to remove transparency from.
        bg_colour (tuple, optional): Background color to use for the transparent areas. Defaults to (255, 255, 255).

    Returns:
        PIL.Image.Image: Image with transparency removed.
    """
    # Only process if image has transparency (http://stackoverflow.com/a/1963146)
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        # Need to convert to RGBA if LA format due to a bug in PIL (http://stackoverflow.com/a/1963146)
        alpha = im.convert("RGBA").split()[-1]

        # Create a new background image of our matt color.
        # Must be RGBA because paste requires both images have the same format
        # (http://stackoverflow.com/a/8720632  and  http://stackoverflow.com/a/9459208)
        bg = Image.new("RGBA", im.size, bg_colour + (255,))
        bg.paste(im, mask=alpha)
        return bg

    else:
        return im


def convert_to_base64(image_url):
    """
    Convert an image, specified by its url, to a PNG and return the base64 encoded string.

    Args:
        image_url (str): URL of the image to convert.

    Returns:
        str: Base64 encoded string of the image.
    """
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))

    # remove transparency
    image = remove_transparency(image)

    with BytesIO() as f:
        image.save(f, format="PNG")
        f.seek(0)

        return base64.b64encode(f.read()).decode("utf-8")


def convert_to_png(image_url):
    response = requests.get(image_url)
    image = Image.open(BytesIO(response.content))

    # remove transparency
    image = remove_transparency(image)

    with BytesIO() as f:
        image.save(f, format="PNG")
        return f.getvalue()

## GPT-4o

In [ ]:
from openai import OpenAI

openai_client = OpenAI()
openai_client.api_key = os.getenv("OPENAI_API_KEY")
model_name = "gpt-4o-2024-08-06"


def get_gpt_caption(image_url, openai_client, prompt, temperature=1.0):
    response = openai_client.responses.create(
        model=model_name,
        input=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": prompt,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{image_url}",
                        "detail": "high",
                    }
                ],
            },
        ],
        text={"format": {"type": "text"}},
        reasoning={},
        tools=[],
        temperature=temperature,
        max_output_tokens=300,
        top_p=1,
        store=False,
    )

    if response.output_text is not None:
        return response.output_text
    else:
        return ""

In [ ]:
for image_index, image_info in enumerate(tqdm(matched_product_df_dict)):
    # check if we already have captions
    if (
        "gpt_caption_orig" in image_info
        and "gpt_caption_rev" in image_info
        and "gpt_caption_rev_2" in image_info
    ):
        print(
            f"Skipping image {image_index} ({image_info['image_url']}) because it already has GPT-4o captions"
        )
        continue

    try:
        image_url = image_info["image_url"]
        image_b64 = convert_to_base64(image_url)

        # get caption from VLM_ORIG_PROMPT
        caption_orig = get_gpt_caption(
            image_b64, openai_client, VLM_ORIG_PROMPT, temperature=1.0
        )
        caption_rev = get_gpt_caption(
            image_b64, openai_client, VLM_REV_PROMPT, temperature=1.0
        )
        caption_rev_2 = get_gpt_caption(
            image_b64, openai_client, VLM_REV_PROMPT_2, temperature=1.0
        )

        # save captions to dataframe
        matched_product_df_dict[image_index]["gpt_caption_orig"] = caption_orig
        matched_product_df_dict[image_index]["gpt_caption_rev"] = caption_rev
        matched_product_df_dict[image_index]["gpt_caption_rev_2"] = caption_rev_2
    except Exception as e:
        print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
        continue
matched_product_df_dict[0]

In [ ]:
# save
os.makedirs("./intermediate/", exist_ok=True)
with open("./intermediate/matched-product-captions-gpt4o.json", "w") as f:
    json.dump(matched_product_df_dict, f)

## Llama

In [ ]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor
from scripts.llama_captioner import generate_caption as get_llama_caption

model_name = "Llama-3.2-11B-Vision-Instruct"
model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

# print model properties
print("Model ID: ", model_id)
print("Device: ", model.device)
print("Dtype: ", model.dtype)

In [ ]:
for image_index, image_info in enumerate(tqdm(matched_product_df_dict)):
    # check if we already have captions
    if (
        "llama_caption_orig" in image_info
        and "llama_caption_rev" in image_info
        and "llama_caption_rev_2" in image_info
    ):
        print(
            f"Skipping image {image_index} ({image_info['image_url']}) because it already has Llama captions"
        )
        continue

    try:
        # load image with pillow
        image = Image.open(io.BytesIO(convert_to_png(image_info["image_url"])))

        # get caption from VLM_ORIG_PROMPT
        caption_orig = get_llama_caption(
            image, model, processor, VLM_ORIG_PROMPT, temperature=1.0
        )
        caption_rev = get_llama_caption(
            image, model, processor, VLM_REV_PROMPT, temperature=1.0
        )
        caption_rev_2 = get_llama_caption(
            image, model, processor, VLM_REV_PROMPT_2, temperature=1.0
        )

        # save captions to dict
        matched_product_df_dict[image_index]["llama_caption_orig"] = caption_orig
        matched_product_df_dict[image_index]["llama_caption_rev"] = caption_rev
        matched_product_df_dict[image_index]["llama_caption_rev_2"] = caption_rev_2
    except Exception as e:
        print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
        continue
matched_product_df_dict[0]

In [ ]:
# save
os.makedirs("./intermediate/", exist_ok=True)
with open("./intermediate/matched-product-captions-llama.json", "w") as f:
    json.dump(matched_product_df_dict, f)

In [ ]:
# clear cache and model objects
torch.cuda.empty_cache()
del model, processor
gc.collect()

## Molmo

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from scripts.molmo_captioner import generate_caption as get_molmo_caption

model_name = "Molmo-7B-D-0924"
model_id = "allenai/Molmo-7B-D-0924"

# load model
processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)

# print model properties
print("Model ID: ", model_id)
print("Device: ", model.device)
print("Dtype: ", model.dtype)

In [ ]:
for image_index, image_info in enumerate(tqdm(matched_product_df_dict)):
    # check if we already have captions
    if (
        "molmo_caption_orig" in image_info
        and "molmo_caption_rev" in image_info
        and "molmo_caption_rev_2" in image_info
    ):
        print(
            f"Skipping image {image_index} ({image_info['image_url']}) because it already has Molmo captions"
        )
        continue

    try:
        # load image with pillow
        image = Image.open(io.BytesIO(convert_to_png(image_info["image_url"])))

        # get caption from VLM_ORIG_PROMPT
        caption_orig = get_molmo_caption(
            image, model, processor, VLM_ORIG_PROMPT, temperature=1.0
        )
        caption_rev = get_molmo_caption(
            image, model, processor, VLM_REV_PROMPT, temperature=1.0
        )
        caption_rev_2 = get_molmo_caption(
            image, model, processor, VLM_REV_PROMPT_2, temperature=1.0
        )

        # save captions to dict
        matched_product_df_dict[image_index]["molmo_caption_orig"] = caption_orig
        matched_product_df_dict[image_index]["molmo_caption_rev"] = caption_rev
        matched_product_df_dict[image_index]["molmo_caption_rev_2"] = caption_rev_2
    except Exception as e:
        print(f"Error processing image {image_index} ({image_info['image_url']}): {e}")
        continue
matched_product_df_dict[0]

In [ ]:
# save
os.makedirs("./intermediate/", exist_ok=True)
with open("./intermediate/matched-product-captions-molmo.json", "w") as f:
    json.dump(matched_product_df_dict, f)

In [ ]:
# clear cache and model objects
torch.cuda.empty_cache()
del model, processor
gc.collect()

## Output final json and CSV

In [ ]:
# write final csv
os.makedirs("./final-output/", exist_ok=True)
with open(
    f"./final-output/matched-product-captions_all-models_{datetime.now().strftime('%Y-%m-%d_%H:%M')}.json",
    "w",
) as f:
    json.dump(matched_product_df_dict, f)

In [ ]:
# output csv
pd.DataFrame(matched_product_df_dict).to_csv(
    f"./final-output/matched-product-captions_all-models_{datetime.now().strftime('%Y-%m-%d_%H:%M')}.csv",
    index=False,
)